<a href="https://colab.research.google.com/github/ShauryaPrakashVerma/AI_Ops/blob/main/Test_Event_Correlation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
Generates a small synthetic alert dataset with KNOWN ground-truth incident
labels, so we can measure how well our correlation formula recovers them.

Each "incident" fires a handful of realistic, topology-connected alerts
within a short time window. We also add pure noise alerts that should NOT
be grouped with anything, to test that the formula doesn't over-merge.
"""

import random
import pandas as pd
import networkx as nx

random.seed(42)

# ---------------------------------------------------------------------------
# 1. Define a small service dependency topology (this is our "known topology")
# ---------------------------------------------------------------------------
topo = nx.DiGraph()
edges = [
    ("web-service", "order-service"),
    ("order-service", "mysql-order-db"),
    ("order-service", "payment-service"),
    ("payment-service", "redis-cache"),
    ("web-service", "auth-service"),
    ("auth-service", "redis-cache"),
]
topo.add_edges_from(edges)
entities = list(topo.nodes())

# ---------------------------------------------------------------------------
# 2. Define incident "templates": a root entity + a short cascade of alert
#    text templates that would realistically follow from it.
# ---------------------------------------------------------------------------
incident_templates = [
    {
        "entities_texts": [
            ("mysql-order-db", "Connection pool exhausted: 200/200 active connections"),
            ("order-service", "Database call timeout after 5000ms"),
            ("web-service", "Upstream request timeout on /checkout"),
        ]
    },
    {
        "entities_texts": [
            ("redis-cache", "Cache node unreachable, connection refused"),
            ("payment-service", "Payment gateway response timeout"),
            ("auth-service", "Session lookup failed, cache miss storm"),
        ]
    },
    {
        "entities_texts": [
            ("payment-service", "High error rate on /charge endpoint: 42%"),
            ("order-service", "Order confirmation failed: downstream 500"),
        ]
    },
    {
        "entities_texts": [
            ("auth-service", "Token validation latency p99 > 3000ms"),
            ("web-service", "Login page error rate spike"),
        ]
    },
]

# ---------------------------------------------------------------------------
# 3. Generate events: for each incident occurrence, fire its alerts within a
#    tight time window; then sprinkle in unrelated noise alerts.
# ---------------------------------------------------------------------------
rows = []
event_id = 0
base_time = 1_700_000_000  # arbitrary unix timestamp
current_time = base_time

NUM_INCIDENT_OCCURRENCES = 18
NUM_NOISE_EVENTS = 25

for i in range(NUM_INCIDENT_OCCURRENCES):
    template = random.choice(incident_templates)
    incident_id = f"incident_{i}"
    incident_start = current_time
    for entity, text in template["entities_texts"]:
        jitter = random.randint(0, 15)  # seconds apart within the same incident
        rows.append({
            "event_id": event_id,
            "entity": entity,
            "text": text,
            "timestamp": incident_start + jitter,
            "true_incident": incident_id,
        })
        event_id += 1
    current_time += random.randint(180, 900)  # gap until next incident

# Common "noisy" alert type that fires constantly and isn't really meaningful
# on its own -- this is what frequency-normalization is supposed to catch.
noisy_texts = [
    "CPU usage high: 91%",
    "CPU usage high: 88%",
    "CPU usage high: 95%",
]

for i in range(NUM_NOISE_EVENTS):
    entity = random.choice(entities)
    text = random.choice(noisy_texts)
    ts = base_time + random.randint(0, current_time - base_time)
    rows.append({
        "event_id": event_id,
        "entity": entity,
        "text": text,
        "timestamp": ts,
        "true_incident": f"noise_{event_id}",  # each noise event is its own "incident" (singleton)
    })
    event_id += 1

df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
df.to_csv("/ablation.csv", index=False)
nx.write_gml(topo, "/topology.gml")

print(f"Generated {len(df)} events across {df['true_incident'].nunique()} true incidents")
print(df.head(10).to_string(index=False))


Generated 73 events across 43 true incidents
 event_id          entity                                                  text  timestamp true_incident
        0  mysql-order-db Connection pool exhausted: 200/200 active connections 1700000000    incident_0
        2     web-service                 Upstream request timeout on /checkout 1700000007    incident_0
        1   order-service                    Database call timeout after 5000ms 1700000008    incident_0
       68     redis-cache                                   CPU usage high: 95% 1700000009      noise_68
       58     redis-cache                                   CPU usage high: 88% 1700000188      noise_58
        4 payment-service                      Payment gateway response timeout 1700000410    incident_1
        3     redis-cache            Cache node unreachable, connection refused 1700000411    incident_1
        5    auth-service               Session lookup failed, cache miss storm 1700000421    incident_1
        6 

In [8]:
"""
Implements the four correlation signals and the weighted composite score.
Each signal can be toggled on/off via weights, which is what makes the
ablation study possible -- "time only" is just weights = (1, 0, 0, 0).
"""

import math
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations


def load_data():
    df = pd.read_csv("/ablation.csv")
    topo = nx.read_gml("/topology.gml")
    return df, topo


def compute_temporal_signal(df, tau=60.0):
    """s_time(i,j) = exp(-|ti - tj| / tau)"""
    n = len(df)
    times = df["timestamp"].values
    S = np.zeros((n, n))
    for i in range(n):
        S[i] = np.exp(-np.abs(times - times[i]) / tau)
    return S


def compute_topology_signal(df, topo, beta=0.6):
    """s_topo(i,j) = beta ^ shortest_path_distance(entity_i, entity_j)
       Uses the undirected version so dependencies count both directions."""
    n = len(df)
    entities = df["entity"].values
    undirected = topo.to_undirected()
    # Precompute all-pairs shortest paths once
    lengths = dict(nx.all_pairs_shortest_path_length(undirected))
    S = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            e_i, e_j = entities[i], entities[j]
            if e_i == e_j:
                S[i, j] = 1.0
            elif e_j in lengths.get(e_i, {}):
                d = lengths[e_i][e_j]
                S[i, j] = beta ** d
            else:
                S[i, j] = 0.0  # not connected in topology at all
    return S


def compute_textual_signal(df):
    """s_text(i,j) = cosine similarity of TF-IDF vectors of alert text."""
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(df["text"].values)
    S = cosine_similarity(tfidf)
    return S


def compute_frequency_signal(df, window_seconds=30):
    """
    s_freq(i,j) = P(A,B) / (P(A) * P(B)), a PMI-style frequency-normalization,
    computed over ALERT TEXT TYPES (not individual events) using sliding
    time windows across the whole dataset. This is the ACOR-inspired term:
    it down-weights alert types that are simply very common.
    """
    texts = df["text"].values
    times = df["timestamp"].values
    n = len(df)

    unique_types = sorted(set(texts))
    type_index = {t: i for i, t in enumerate(unique_types)}
    n_types = len(unique_types)

    # Build sliding windows over the sorted timeline
    order = np.argsort(times)
    sorted_times = times[order]
    sorted_texts = texts[order]

    window_type_sets = []
    start = 0
    for end in range(n):
        while sorted_times[end] - sorted_times[start] > window_seconds:
            start += 1
        window_type_sets.append(set(sorted_texts[start:end + 1]))

    total_windows = n
    type_count = np.zeros(n_types)
    pair_count = np.zeros((n_types, n_types))

    for w in window_type_sets:
        idxs = [type_index[t] for t in w]
        for idx in idxs:
            type_count[idx] += 1
        for a, b in combinations(idxs, 2):
            pair_count[a, b] += 1
            pair_count[b, a] += 1
        for a in idxs:
            pair_count[a, a] += 1  # same-type co-occurrence counts as max correlation

    P_type = type_count / total_windows
    S_type = np.zeros((n_types, n_types))
    for a in range(n_types):
        for b in range(n_types):
            p_ab = pair_count[a, b] / total_windows
            denom = P_type[a] * P_type[b]
            S_type[a, b] = (p_ab / denom) if denom > 0 else 0.0

    # normalize into [0,1] range for comparability with the other signals
    max_val = S_type.max() if S_type.max() > 0 else 1.0
    S_type = S_type / max_val

    # map back onto individual events
    type_ids = np.array([type_index[t] for t in texts])
    S = S_type[np.ix_(type_ids, type_ids)]
    return S


def combine_signals(S_time, S_topo, S_text, S_freq, weights):
    w_time, w_topo, w_text, w_freq = weights
    total = w_time + w_topo + w_text + w_freq
    if total == 0:
        raise ValueError("At least one weight must be non-zero")
    Score = (w_time * S_time + w_topo * S_topo + w_text * S_text + w_freq * S_freq) / total
    return Score


In [9]:
"""
Runs the ablation study: builds a similarity graph from the Score matrix,
clusters via connected components, and evaluates against the KNOWN
ground-truth incidents using pairwise precision / recall / F1.

Pairwise metric definition (standard for clustering evaluation):
  For every pair of events (i, j):
    - "positive" if they are predicted to be in the same incident
    - "true positive" if they are ALSO actually in the same true incident
  Precision = TP / (TP + FP)   -> of the pairs we grouped, how many were correct
  Recall    = TP / (TP + FN)   -> of the pairs that should be grouped, how many did we catch
  F1        = harmonic mean of the two
"""

import numpy as np
import pandas as pd
import networkx as nx
from itertools import combinations

from importlib import import_module
import sys
sys.path.insert(0, "/ablation")
formula = import_module("02_correlation_formula".replace("-", "_"))


def cluster_from_score(Score, threshold):
    n = Score.shape[0]
    g = nx.Graph()
    g.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 1, n):
            if Score[i, j] >= threshold:
                g.add_edge(i, j)
    components = list(nx.connected_components(g))
    cluster_id = np.zeros(n, dtype=int)
    for cid, comp in enumerate(components):
        for idx in comp:
            cluster_id[idx] = cid
    return cluster_id


def pairwise_prf1(true_labels, pred_labels):
    n = len(true_labels)
    tp = fp = fn = tn = 0
    for i, j in combinations(range(n), 2):
        same_true = true_labels[i] == true_labels[j]
        same_pred = pred_labels[i] == pred_labels[j]
        if same_pred and same_true:
            tp += 1
        elif same_pred and not same_true:
            fp += 1
        elif not same_pred and same_true:
            fn += 1
        else:
            tn += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def run_config(name, weights, S_time, S_topo, S_text, S_freq, true_labels, thresholds):
    """
    Sweeps thresholds and reports the BEST F1 found for this configuration.
    This is necessary because each signal (or combination) produces scores on
    a different scale -- a single fixed threshold would unfairly penalize
    whichever configuration happens to produce lower average scores, rather
    than reflecting how good the signal actually is.
    """
    Score = formula.combine_signals(S_time, S_topo, S_text, S_freq, weights)
    best = {"precision": 0, "recall": 0, "f1": -1, "threshold": None, "predicted_incidents": None}
    for t in thresholds:
        pred_labels = cluster_from_score(Score, t)
        precision, recall, f1 = pairwise_prf1(true_labels, pred_labels)
        if f1 > best["f1"]:
            best = {
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "threshold": t,
                "predicted_incidents": len(set(pred_labels)),
            }
    return {
        "config": name,
        "weights (time, topo, text, freq)": weights,
        "best_threshold": round(best["threshold"], 2),
        "precision": round(best["precision"], 3),
        "recall": round(best["recall"], 3),
        "f1": round(best["f1"], 3),
        "predicted_incidents": best["predicted_incidents"],
    }


if __name__ == "__main__":
    df, topo = formula.load_data()
    true_labels = df["true_incident"].values

    print("Computing signals...")
    S_time = formula.compute_temporal_signal(df, tau=60.0)
    S_topo = formula.compute_topology_signal(df, topo, beta=0.6)
    S_text = formula.compute_textual_signal(df)
    S_freq = formula.compute_frequency_signal(df, window_seconds=30)

    THRESHOLDS = np.arange(0.05, 0.96, 0.05)

    configs = [
        ("Full hybrid (all 4 signals)", (0.25, 0.25, 0.25, 0.25)),
        ("Time only",                   (1.0, 0.0, 0.0, 0.0)),
        ("Topology only",               (0.0, 1.0, 0.0, 0.0)),
        ("Text only",                   (0.0, 0.0, 1.0, 0.0)),
        ("Frequency only",              (0.0, 0.0, 0.0, 1.0)),
    ]

    results = []
    for name, weights in configs:
        results.append(run_config(name, weights, S_time, S_topo, S_text, S_freq, true_labels, THRESHOLDS))

    results_df = pd.DataFrame(results)
    results_df.to_csv("/home/claude/ablation/ablation_results.csv", index=False)
    print(results_df.to_string(index=False))


ModuleNotFoundError: No module named '02_correlation_formula'

In [ ]:
"""
The ablation study revealed that naive EQUAL weights (0.25 each) make the
hybrid formula perform *worse* than the best single signal. This script
investigates why, and searches for better weights -- which is exactly the
"weight sensitivity analysis" task, just done as a full grid search instead
of manual tuning.
"""

import numpy as np
import pandas as pd
from importlib import import_module
import sys
sys.path.insert(0, "/home/claude/ablation")

formula = import_module("02_correlation_formula")
ablation = import_module("03_ablation_study")

df, topo = formula.load_data()
true_labels = df["true_incident"].values

S_time = formula.compute_temporal_signal(df, tau=60.0)
S_topo = formula.compute_topology_signal(df, topo, beta=0.6)
S_text = formula.compute_textual_signal(df)
S_freq = formula.compute_frequency_signal(df, window_seconds=30)

THRESHOLDS = np.arange(0.05, 0.96, 0.05)

# Grid search over weight combinations (steps of 0.25, summing to 1.0)
step = 0.25
weight_values = np.arange(0.0, 1.01, step)
grid_results = []

for w_time in weight_values:
    for w_topo in weight_values:
        for w_text in weight_values:
            for w_freq in weight_values:
                total = w_time + w_topo + w_text + w_freq
                if abs(total - 1.0) > 1e-6:
                    continue  # only keep combinations that sum to 1
                weights = (w_time, w_topo, w_text, w_freq)
                result = ablation.run_config(
                    "grid", weights, S_time, S_topo, S_text, S_freq, true_labels, THRESHOLDS
                )
                result["weights"] = weights
                grid_results.append(result)

grid_df = pd.DataFrame(grid_results).sort_values("f1", ascending=False)
grid_df.to_csv("/home/claude/ablation/weight_search_results.csv", index=False)

print("Top 5 weight combinations by F1:")
print(grid_df[["weights", "best_threshold", "precision", "recall", "f1"]].head(5).to_string(index=False))

print("\nFor comparison, equal weights (0.25, 0.25, 0.25, 0.25):")
equal_row = grid_df[grid_df["weights"].apply(lambda w: w == (0.25, 0.25, 0.25, 0.25))]
print(equal_row[["weights", "best_threshold", "precision", "recall", "f1"]].to_string(index=False))

print("\nFor comparison, best single signal (time only) from the ablation study:")
print("weights=(1.0, 0.0, 0.0, 0.0)  f1=0.923  (see ablation_results.csv)")
